In [1]:
from pathlib import Path
import copy
import os
import sys

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

PROJECT_ROOT = Path('/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from DMTimeShardDataset import DMTimeShardDataset
from moe.train_joint_ensemble import build_joint_model
from training_utils import label_encoding

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cuda


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASELINE_REJECTION_DIR = PROJECT_ROOT / 'final_checkpoints' / 'baseline_rejection_ensemble'
MOE_CHECKPOINT = PROJECT_ROOT / 'final_checkpoints' / 'joint_cascade_moe_best.pth'

dataset_cfg = {
    'output_dir': '/cephfs/users/oleksjuk/MA/WP2-1/DM_time_dataset_creator/outputs',
    'prefix': 'B0531+21_59000_48386',
}

baseline_joint_config = {
    'dataset': dataset_cfg,
    'model': {
        'temperature': 1.0,
        'f_small': {
            'model_name': 'DM_time_binary_classificator_241002_3_GAP',
            'resolution': 256,
            'mode': 'dmt',
            'dropout': False,
            'checkpoint': str(BASELINE_REJECTION_DIR / 'prot-DM_time_binary_classificator_241002_3_GAP-014-0.764-0.740.pth'),
        },
        'f_mid': {
            'model_name': 'DM_time_binary_classificator_241002_5_GAP',
            'resolution': 256,
            'mode': 'ft',
            'dropout': False,
            'checkpoint': str(BASELINE_REJECTION_DIR / 'prot-DM_time_binary_classificator_241002_5_GAP-060-0.973-0.948.pth'),
        },
        'f_large': {
            'model_name': 'DM_time_binary_classificator_resnet18',
            'resolution': 256,
            'mode': 'dmft',
            'dropout': False,
            'checkpoint': str(BASELINE_REJECTION_DIR / 'prot-DM_time_binary_classificator_resnet18-003-0.993-0.993.pth'),
        },
        'r1': {
            'model_name': 'conv_mlp',
            'cnn_channels': 64,
            'extra_conv': False,
            'pool_size': 7,
            'hidden_dim': 64,
            'dropout': 0.0,
            'checkpoint': str(BASELINE_REJECTION_DIR / 'prot-run_embedding_3_GAP_conv_mlp_lr1.05e-05_wd0.00e+00_drop0.0_channels64_extraFalse_pool7_hidden64_worker2_trial3-042-0.712-0.635.pth'),
        },
        'r2': {
            'model_name': 'conv_mlp',
            'cnn_channels': 64,
            'extra_conv': True,
            'pool_size': 7,
            'hidden_dim': 128,
            'dropout': 0.2,
            'checkpoint': str(BASELINE_REJECTION_DIR / 'prot-run_embedding_r2_conv_mlp_lr4.51e-05_wd0.00e+00_drop0.2_channels64_extraTrue_pool7_hidden128_worker13_trial6-035-0.825-0.842.pth'),
        },
    },
}


In [ ]:
print('Loading baseline rejection ensemble...')
baseline_model = build_joint_model(baseline_joint_config, device).eval()

print('Loading joint training MoE...')
moe_ckpt = torch.load(MOE_CHECKPOINT, map_location='cpu')
moe_config = copy.deepcopy(moe_ckpt['config'])
moe_config['dataset']['output_dir'] = dataset_cfg['output_dir']
moe_config['dataset']['prefix'] = dataset_cfg['prefix']

moe_model = build_joint_model(moe_config, device)
moe_model.load_state_dict(moe_ckpt['model_state_dict'])
moe_model.eval()

print(f'Loaded MoE checkpoint: {MOE_CHECKPOINT}')
print(f'Best epoch: {moe_ckpt.get("epoch")}')
if 'val_topk/accuracy' in moe_ckpt.get('metrics', {}):
    print(f'Checkpoint val top-k accuracy: {moe_ckpt["metrics"]["val_topk/accuracy"]:.4f}')

EVAL_SPLIT = 'test'  # 'train', 'val', or 'test'
print(f'Loading {EVAL_SPLIT} dataset...')
pulse_val_dataset = DMTimeShardDataset(dataset_cfg, use_freq_time=True, split=EVAL_SPLIT)
pulse_val_dataset.labels = label_encoding(pulse_val_dataset.labels.astype(object))

pulse_val_loader = DataLoader(
    pulse_val_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=2,
)
print(f'{EVAL_SPLIT} samples: {len(pulse_val_dataset)}')



In [ ]:
def append_outputs(prefix, outputs, labels):
    expert_logits = outputs['expert_logits']
    collected[f'{prefix}_r1_scores'].append(outputs['rejector_probs']['r1'].detach().cpu())
    collected[f'{prefix}_r2_scores'].append(outputs['rejector_probs']['r2'].detach().cpu())
    collected[f'{prefix}_small_preds'].append(expert_logits[:, 0].argmax(dim=1).detach().cpu())
    collected[f'{prefix}_mid_preds'].append(expert_logits[:, 1].argmax(dim=1).detach().cpu())
    collected[f'{prefix}_large_preds'].append(expert_logits[:, 2].argmax(dim=1).detach().cpu())


print('Computing predictions and rejector scores...')
collected = {key: [] for key in (
    'baseline_r1_scores', 'baseline_r2_scores',
    'baseline_small_preds', 'baseline_mid_preds', 'baseline_large_preds',
    'moe_r1_scores', 'moe_r2_scores',
    'moe_small_preds', 'moe_mid_preds', 'moe_large_preds',
)}
all_labels = []

with torch.no_grad():
    for batch in tqdm(pulse_val_loader):
        labels = batch['label'].to(device)
        all_labels.append(labels.detach().cpu())

        baseline_outputs = baseline_model._forward_all_aux(batch)
        append_outputs('baseline', baseline_outputs, labels)
        del baseline_outputs

        moe_outputs = moe_model._forward_all_aux(batch)
        append_outputs('moe', moe_outputs, labels)
        del moe_outputs

labels_np = torch.cat(all_labels).numpy()

baseline_r1_scores_np = torch.cat(collected['baseline_r1_scores']).numpy()
baseline_r2_scores_np = torch.cat(collected['baseline_r2_scores']).numpy()
baseline_small_preds_np = torch.cat(collected['baseline_small_preds']).numpy()
baseline_mid_preds_np = torch.cat(collected['baseline_mid_preds']).numpy()
baseline_large_preds_np = torch.cat(collected['baseline_large_preds']).numpy()

r1_scores_np = torch.cat(collected['moe_r1_scores']).numpy()
r2_scores_np = torch.cat(collected['moe_r2_scores']).numpy()
small_preds_np = torch.cat(collected['moe_small_preds']).numpy()
mid_preds_np = torch.cat(collected['moe_mid_preds']).numpy()
large_preds_np = torch.cat(collected['moe_large_preds']).numpy()

print(f'Collected {len(labels_np)} samples')
print(f'Baseline small/mid/large acc: {np.mean(baseline_small_preds_np == labels_np):.4f} / {np.mean(baseline_mid_preds_np == labels_np):.4f} / {np.mean(baseline_large_preds_np == labels_np):.4f}')
print(f'MoE small/mid/large acc:      {np.mean(small_preds_np == labels_np):.4f} / {np.mean(mid_preds_np == labels_np):.4f} / {np.mean(large_preds_np == labels_np):.4f}')


In [ ]:
# ========== Grid Search over R1 and R2 Reject Rates ==========
r1_reject_rates = np.linspace(0.0, 1.0, 21)
r2_conditional_reject_rates = np.linspace(0.0, 1.0, 21)


def topk_reject_mask(scores, reject_rate):
    scores = np.asarray(scores)
    n = scores.shape[0]
    k = int(np.rint(float(reject_rate) * n))
    k = int(np.clip(k, 0, n))

    mask = np.zeros(n, dtype=bool)
    if k == 0:
        return mask, float('inf'), 0.0
    if k == n:
        mask[:] = True
        return mask, float('-inf'), 1.0

    top_idx = np.argpartition(scores, n - k)[n - k:]
    mask[top_idx] = True
    threshold = float(scores[top_idx].min())
    return mask, threshold, float(mask.mean())


def compute_accuracy_grid_by_reject_rates(r1_scores, r2_scores, small_preds, mid_preds, large_preds, labels):
    accuracy = np.zeros((len(r1_reject_rates), len(r2_conditional_reject_rates)))
    r1_threshold_grid = np.zeros_like(accuracy)
    r2_threshold_grid = np.zeros_like(accuracy)
    actual_r1_rate_grid = np.zeros_like(accuracy)
    actual_r2_rate_grid = np.zeros_like(accuracy)

    for i, r1_rate in enumerate(r1_reject_rates):
        r1_reject_mask, r1_threshold, actual_r1_rate = topk_reject_mask(r1_scores, r1_rate)
        r1_rejected_indices = np.flatnonzero(r1_reject_mask)

        for j, r2_rate in enumerate(r2_conditional_reject_rates):
            r2_reject_mask = np.zeros_like(r1_reject_mask)
            r2_threshold = float('inf')
            actual_r2_rate = 0.0

            if len(r1_rejected_indices) > 0:
                r2_local_mask, r2_threshold, actual_r2_rate = topk_reject_mask(
                    r2_scores[r1_rejected_indices],
                    r2_rate,
                )
                r2_reject_mask[r1_rejected_indices[r2_local_mask]] = True

            r2_accept_mask = r1_reject_mask & ~r2_reject_mask
            y_pred = small_preds.copy()
            y_pred[r2_accept_mask] = mid_preds[r2_accept_mask]
            y_pred[r2_reject_mask] = large_preds[r2_reject_mask]

            accuracy[i, j] = (y_pred == labels).mean()
            r1_threshold_grid[i, j] = r1_threshold
            r2_threshold_grid[i, j] = r2_threshold
            actual_r1_rate_grid[i, j] = actual_r1_rate
            actual_r2_rate_grid[i, j] = actual_r2_rate

    return {
        'accuracy': accuracy,
        'r1_threshold': r1_threshold_grid,
        'r2_threshold': r2_threshold_grid,
        'actual_r1_rate': actual_r1_rate_grid,
        'actual_r2_rate': actual_r2_rate_grid,
    }


print('Computing accuracy grids by reject rates...')
baseline_grid = compute_accuracy_grid_by_reject_rates(
    baseline_r1_scores_np,
    baseline_r2_scores_np,
    baseline_small_preds_np,
    baseline_mid_preds_np,
    baseline_large_preds_np,
    labels_np,
)

moe_grid = compute_accuracy_grid_by_reject_rates(
    r1_scores_np,
    r2_scores_np,
    small_preds_np,
    mid_preds_np,
    large_preds_np,
    labels_np,
)

baseline_accuracy_grid = baseline_grid['accuracy']
accuracy_grid = moe_grid['accuracy']
accuracy_diff_grid = accuracy_grid - baseline_accuracy_grid

R1_grid, R2_grid = np.meshgrid(r1_reject_rates, r2_conditional_reject_rates, indexing='ij')
print('Accuracy grids computed!')


In [ ]:
# ========== Rejector Comparison at Own Operating Points ==========
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

BASELINE_R1_THRESHOLD = 0.548808
BASELINE_R2_THRESHOLD = 0.381863

MOE_R1_THRESHOLD = 0.5663751363754272
MOE_R2_THRESHOLD = 0.8648102879524231

print(f'Baseline thresholds: R1={BASELINE_R1_THRESHOLD:.6f}, R2={BASELINE_R2_THRESHOLD:.6f}')
print(f'Joint MoE thresholds: R1={MOE_R1_THRESHOLD:.6f}, R2={MOE_R2_THRESHOLD:.6f}')
if globals().get('EVAL_SPLIT', 'val') != 'val':
    print(f'Thresholds were selected on val; reject rates below are measured on {EVAL_SPLIT}.')


def safe_score_metrics(y_true, scores):
    y_true = np.asarray(y_true, dtype=bool)
    scores = np.asarray(scores, dtype=float)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return {'ap': np.nan, 'roc_auc': np.nan, 'positive_rate': float(y_true.mean()) if len(y_true) else np.nan}
    return {
        'ap': average_precision_score(y_true.astype(int), scores),
        'roc_auc': roc_auc_score(y_true.astype(int), scores),
        'positive_rate': float(y_true.mean()),
    }


def decision_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=bool)
    y_pred = np.asarray(y_pred, dtype=bool)
    tp = int(np.logical_and(y_true, y_pred).sum())
    fp = int(np.logical_and(~y_true, y_pred).sum())
    fn = int(np.logical_and(y_true, ~y_pred).sum())
    tn = int(np.logical_and(~y_true, ~y_pred).sum())

    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else np.nan
    false_accept_rate = fn / (tp + fn) if tp + fn else np.nan
    false_reject_rate = fp / (fp + tn) if fp + tn else np.nan
    return {
        'precision_at_threshold': precision,
        'recall_at_threshold': recall,
        'f1_at_threshold': f1,
        'false_accept_rate': false_accept_rate,
        'false_reject_rate': false_reject_rate,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn,
    }


def build_threshold_masks(r1_scores, r2_scores, r1_threshold, r2_threshold):
    r1_mask = r1_scores >= r1_threshold
    r1_indices = np.flatnonzero(r1_mask)

    r2_mask = np.zeros_like(r1_mask, dtype=bool)
    r2_local_mask = np.zeros(0, dtype=bool)
    if len(r1_indices) > 0:
        r2_local_mask = r2_scores[r1_indices] >= r2_threshold
        r2_mask[r1_indices[r2_local_mask]] = True

    return {
        'r1_mask': r1_mask,
        'r2_mask': r2_mask,
        'r1_indices': r1_indices,
        'r2_local_mask': r2_local_mask,
        'r1_threshold': float(r1_threshold),
        'r2_threshold': float(r2_threshold),
        'r1_reject_rate': float(r1_mask.mean()),
        'r2_conditional_reject_rate': float(r2_local_mask.mean()) if len(r1_indices) > 0 else 0.0,
    }


def compare_rejectors(name, r1_scores, r2_scores, small_preds, mid_preds, large_preds, r1_threshold, r2_threshold):
    masks = build_threshold_masks(r1_scores, r2_scores, r1_threshold, r2_threshold)
    r1_mask = masks['r1_mask']
    r2_mask = masks['r2_mask']
    r1_indices = masks['r1_indices']
    r2_local_mask = masks['r2_local_mask']

    small_wrong = small_preds != labels_np
    mid_wrong = mid_preds != labels_np
    large_correct = large_preds == labels_np
    useful_forward = small_wrong & ((mid_preds == labels_np) | large_correct)
    useful_large = mid_wrong & large_correct

    y_pred = small_preds.copy()
    r2_accept_mask = r1_mask & ~r2_mask
    y_pred[r2_accept_mask] = mid_preds[r2_accept_mask]
    y_pred[r2_mask] = large_preds[r2_mask]

    route_row = {
        'ensemble': name,
        'r1_reject_rate': masks['r1_reject_rate'],
        'r2_conditional_reject_rate': masks['r2_conditional_reject_rate'],
        'small_usage': float((~r1_mask).mean()),
        'mid_usage': float(r2_accept_mask.mean()),
        'large_usage': float(r2_mask.mean()),
        'r1_threshold': masks['r1_threshold'],
        'r2_threshold': masks['r2_threshold'],
        'cascade_accuracy': float((y_pred == labels_np).mean()),
        'r1_missed_small_errors': int(np.logical_and(small_wrong, ~r1_mask).sum()),
        'r1_unnecessary_forwards': int(np.logical_and(~small_wrong, r1_mask).sum()),
        'r2_missed_useful_large': int(np.logical_and.reduce((useful_large, r1_mask, ~r2_mask)).sum()),
        'r2_useful_large_taken': int(np.logical_and(useful_large, r2_mask).sum()),
        'r2_harmful_large': int(np.logical_and.reduce((mid_preds == labels_np, large_preds != labels_np, r2_mask)).sum()),
        'r2_wasted_large': int(np.logical_and(mid_preds == labels_np, r2_mask).sum()),
    }

    rows = []
    for rejector, target_name, target, scores, pred_mask in [
        ('R1', 'small wrong', small_wrong, r1_scores, r1_mask),
        ('R1', 'useful forward', useful_forward, r1_scores, r1_mask),
    ]:
        rows.append({
            'ensemble': name,
            'rejector': rejector,
            'target': target_name,
            'n': len(target),
            'threshold': masks['r1_threshold'],
            'reject_rate': masks['r1_reject_rate'],
            **safe_score_metrics(target, scores),
            **decision_metrics(target, pred_mask),
        })

    if len(r1_indices) > 0:
        for target_name, target in [
            ('mid wrong', mid_wrong[r1_indices]),
            ('useful large', useful_large[r1_indices]),
        ]:
            rows.append({
                'ensemble': name,
                'rejector': 'R2',
                'target': target_name,
                'n': len(target),
                'threshold': masks['r2_threshold'],
                'reject_rate': masks['r2_conditional_reject_rate'],
                **safe_score_metrics(target, r2_scores[r1_indices]),
                **decision_metrics(target, r2_local_mask),
            })

    return rows, route_row


baseline_rejector_rows, baseline_route_row = compare_rejectors(
    'Baseline',
    baseline_r1_scores_np,
    baseline_r2_scores_np,
    baseline_small_preds_np,
    baseline_mid_preds_np,
    baseline_large_preds_np,
    BASELINE_R1_THRESHOLD,
    BASELINE_R2_THRESHOLD,
)
moe_rejector_rows, moe_route_row = compare_rejectors(
    'Joint training MoE',
    r1_scores_np,
    r2_scores_np,
    small_preds_np,
    mid_preds_np,
    large_preds_np,
    MOE_R1_THRESHOLD,
    MOE_R2_THRESHOLD,
)

rejector_metrics_df = pd.DataFrame(baseline_rejector_rows + moe_rejector_rows)
routing_error_df = pd.DataFrame([baseline_route_row, moe_route_row])

threshold_summary_df = routing_error_df[[
    'ensemble', 'r1_threshold', 'r2_threshold', 'r1_reject_rate',
    'r2_conditional_reject_rate', 'small_usage', 'mid_usage', 'large_usage',
    'cascade_accuracy',
]]

display(threshold_summary_df)
display(rejector_metrics_df)
display(routing_error_df)





In [ ]:
# ========== 3D Surface Plot ==========
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(R1_grid, R2_grid, accuracy_grid, cmap='viridis', alpha=0.8, edgecolor='none')

ax.set_xlabel('R1 Reject Rate', fontsize=11)
ax.set_ylabel('R2 Conditional Reject Rate', fontsize=11)
ax.set_zlabel('Validation Accuracy', fontsize=11)
ax.set_title('Joint Training MoE Accuracy vs Reject Rates', fontsize=13, fontweight='bold')
fig.colorbar(surf, ax=ax, pad=0.1, label='Accuracy')
ax.view_init(elev=25, azim=45)

plt.tight_layout()
plt.show()

best_idx = np.unravel_index(np.argmax(accuracy_grid), accuracy_grid.shape)
best_r1_rate = r1_reject_rates[best_idx[0]]
best_r2_rate = r2_conditional_reject_rates[best_idx[1]]
best_acc = accuracy_grid[best_idx]
baseline_acc_at_moe_best = baseline_accuracy_grid[best_idx]

print('=== Best MoE Configuration by Reject Rates ===')
print(f'R1 Reject Rate: {best_r1_rate:.4f} (actual {moe_grid["actual_r1_rate"][best_idx]:.4f})')
print(f'R2 Conditional Reject Rate: {best_r2_rate:.4f} (actual {moe_grid["actual_r2_rate"][best_idx]:.4f})')
print(f'MoE R1 Threshold: {moe_grid["r1_threshold"][best_idx]:.6f}')
print(f'MoE R2 Threshold: {moe_grid["r2_threshold"][best_idx]:.6f}')
print(f'Baseline R1 Threshold at same rates: {baseline_grid["r1_threshold"][best_idx]:.6f}')
print(f'Baseline R2 Threshold at same rates: {baseline_grid["r2_threshold"][best_idx]:.6f}')
print(f'MoE Validation Accuracy: {best_acc:.4f}')
print(f'Baseline Accuracy at same reject rates: {baseline_acc_at_moe_best:.4f}')
print(f'Difference MoE - Baseline: {best_acc - baseline_acc_at_moe_best:+.4f}')

fig, ax = plt.subplots(figsize=(10, 8))
contour = ax.contourf(R1_grid, R2_grid, accuracy_grid, levels=20, cmap='viridis')
contour_lines = ax.contour(R1_grid, R2_grid, accuracy_grid, levels=10, colors='black', alpha=0.3, linewidths=0.5)
ax.clabel(contour_lines, inline=True, fontsize=8)
ax.scatter([best_r1_rate], [best_r2_rate], color='red', s=200, marker='*', edgecolors='white', linewidths=2, label=f'Best: Acc={best_acc:.4f}')

ax.set_xlabel('R1 Reject Rate', fontsize=12)
ax.set_ylabel('R2 Conditional Reject Rate', fontsize=12)
ax.set_title('Contour Plot: Joint Training MoE Accuracy', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
fig.colorbar(contour, ax=ax, label='Accuracy')
plt.tight_layout()
plt.show()


In [ ]:
# ========== Difference Plot: MoE Accuracy - Baseline Accuracy ==========
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

limit = max(float(np.nanmax(np.abs(accuracy_diff_grid))), 1e-6)
surf_diff = ax.plot_surface(
    R1_grid,
    R2_grid,
    accuracy_diff_grid,
    cmap='coolwarm',
    vmin=-limit,
    vmax=limit,
    alpha=0.85,
    edgecolor='none',
)
ax.plot_surface(R1_grid, R2_grid, np.zeros_like(accuracy_diff_grid), color='black', alpha=0.12, edgecolor='none')

ax.set_xlabel('R1 Reject Rate', fontsize=11)
ax.set_ylabel('R2 Conditional Reject Rate', fontsize=11)
ax.set_zlabel('Accuracy Difference', fontsize=11)
ax.set_title('MoE - Baseline Accuracy Difference at Equal Reject Rates', fontsize=13, fontweight='bold')
fig.colorbar(surf_diff, ax=ax, pad=0.1, label='MoE accuracy - baseline accuracy')
ax.view_init(elev=25, azim=45)
plt.tight_layout()
plt.show()

best_diff_idx = np.unravel_index(np.argmax(accuracy_diff_grid), accuracy_diff_grid.shape)
worst_diff_idx = np.unravel_index(np.argmin(accuracy_diff_grid), accuracy_diff_grid.shape)
print('=== Biggest MoE Advantage at Equal Reject Rates ===')
print(f'R1 Reject Rate: {r1_reject_rates[best_diff_idx[0]]:.4f}')
print(f'R2 Conditional Reject Rate: {r2_conditional_reject_rates[best_diff_idx[1]]:.4f}')
print(f'MoE R1/R2 Thresholds: {moe_grid["r1_threshold"][best_diff_idx]:.6f} / {moe_grid["r2_threshold"][best_diff_idx]:.6f}')
print(f'Baseline R1/R2 Thresholds: {baseline_grid["r1_threshold"][best_diff_idx]:.6f} / {baseline_grid["r2_threshold"][best_diff_idx]:.6f}')
print(f'MoE Accuracy: {accuracy_grid[best_diff_idx]:.4f}')
print(f'Baseline Accuracy: {baseline_accuracy_grid[best_diff_idx]:.4f}')
print(f'Difference: {accuracy_diff_grid[best_diff_idx]:+.4f}')

print('=== Biggest Baseline Advantage at Equal Reject Rates ===')
print(f'R1 Reject Rate: {r1_reject_rates[worst_diff_idx[0]]:.4f}')
print(f'R2 Conditional Reject Rate: {r2_conditional_reject_rates[worst_diff_idx[1]]:.4f}')
print(f'MoE R1/R2 Thresholds: {moe_grid["r1_threshold"][worst_diff_idx]:.6f} / {moe_grid["r2_threshold"][worst_diff_idx]:.6f}')
print(f'Baseline R1/R2 Thresholds: {baseline_grid["r1_threshold"][worst_diff_idx]:.6f} / {baseline_grid["r2_threshold"][worst_diff_idx]:.6f}')
print(f'MoE Accuracy: {accuracy_grid[worst_diff_idx]:.4f}')
print(f'Baseline Accuracy: {baseline_accuracy_grid[worst_diff_idx]:.4f}')
print(f'Difference: {accuracy_diff_grid[worst_diff_idx]:+.4f}')

fig, ax = plt.subplots(figsize=(10, 8))
contour_diff = ax.contourf(
    R1_grid,
    R2_grid,
    accuracy_diff_grid,
    levels=21,
    cmap='coolwarm',
    vmin=-limit,
    vmax=limit,
)
zero_line = ax.contour(R1_grid, R2_grid, accuracy_diff_grid, levels=[0.0], colors='black', linewidths=1.5)
ax.clabel(zero_line, inline=True, fontsize=8, fmt='0')

ax.scatter([r1_reject_rates[best_diff_idx[0]]], [r2_conditional_reject_rates[best_diff_idx[1]]], color='green', s=160, marker='*', edgecolors='white', linewidths=1.5, label='max MoE advantage')
ax.scatter([r1_reject_rates[worst_diff_idx[0]]], [r2_conditional_reject_rates[worst_diff_idx[1]]], color='purple', s=120, marker='X', edgecolors='white', linewidths=1.5, label='max baseline advantage')
ax.set_xlabel('R1 Reject Rate', fontsize=12)
ax.set_ylabel('R2 Conditional Reject Rate', fontsize=12)
ax.set_title('Contour Plot: MoE - Baseline Accuracy', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
fig.colorbar(contour_diff, ax=ax, label='Accuracy Difference')
plt.tight_layout()
plt.show()


In [ ]:
# ========== Common Latency Grid at Equal Reject Rates ==========
actual_r1_rate_grid = moe_grid['actual_r1_rate']
actual_r2_rate_grid = moe_grid['actual_r2_rate']
latency_grid = (
    1 * 0.21
    + 1 * 0.25
    + (1 - actual_r1_rate_grid) * 0.13
    + actual_r1_rate_grid * (
        0.51
        + 0.21
        + 0.38
        + ((1 - actual_r2_rate_grid) * 0.10)
        + actual_r2_rate_grid * 2.48
    )
)

print('Common latency grid computed!')

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')
surf_lat = ax.plot_surface(R1_grid, R2_grid, latency_grid, cmap='plasma', alpha=0.8, edgecolor='none')

ax.set_xlabel('R1 Reject Rate', fontsize=11)
ax.set_ylabel('R2 Conditional Reject Rate', fontsize=11)
ax.set_zlabel('System Latency (ms)', fontsize=11)
ax.set_title('System Latency vs Reject Rates', fontsize=13, fontweight='bold')
fig.colorbar(surf_lat, ax=ax, pad=0.1, label='Latency')
ax.view_init(elev=25, azim=45)

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))
contour_lat = ax.contourf(R1_grid, R2_grid, latency_grid, levels=20, cmap='plasma')
contour_lat_lines = ax.contour(R1_grid, R2_grid, latency_grid, levels=10, colors='black', alpha=0.3, linewidths=0.5)
ax.clabel(contour_lat_lines, inline=True, fontsize=8)

ax.set_xlabel('R1 Reject Rate', fontsize=12)
ax.set_ylabel('R2 Conditional Reject Rate', fontsize=12)
ax.set_title('Contour Plot: System Latency', fontsize=13, fontweight='bold')
fig.colorbar(contour_lat, ax=ax, label='Latency')
plt.tight_layout()
plt.show()


In [ ]:
# ========== Accuracy vs Latency ==========
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(latency_grid.ravel(), baseline_accuracy_grid.ravel(), s=28, alpha=0.55, label='Baseline')
ax.scatter(latency_grid.ravel(), accuracy_grid.ravel(), s=28, alpha=0.55, label='Joint training MoE')

ax.set_xlabel('System Latency (ms)', fontsize=12)
ax.set_ylabel('Validation Accuracy', fontsize=12)
ax.set_title('Accuracy vs Common Latency', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()
